# Colon Cancer Detection: train, evaluate and publish the live demo

Run this notebook top to bottom on Google Colab. It takes about 20 minutes.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

What it does:
1. Downloads the LC25000 colon images from Kaggle
2. Trains a ResNet-18 with CLAHE preprocessing
3. Evaluates on the held-out test set and prints your real results
4. Publishes a live demo to Hugging Face Spaces (free)

For step 4 you need a free Hugging Face account and a token with **write** access: huggingface.co → Settings → Access Tokens → Create new token → Write.

## 1. Get the code

In [ ]:
!git clone -q https://github.com/gyanasaiadusumilli/colon-cancer-detection.git
%cd colon-cancer-detection
!pip install -q kagglehub huggingface_hub
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none. Switch the runtime to T4 GPU first.")

## 2. Download the dataset
If Kaggle asks you to sign in, a login box appears. Paste your Kaggle username and API key (kaggle.com → Settings → API → Create New Token).

In [ ]:
import kagglehub
from pathlib import Path

DATASET = "andrewmvd/lung-and-colon-cancer-histopathological-images"
try:
    root = kagglehub.dataset_download(DATASET)
except Exception as e:
    print("Kaggle needs you to sign in:", e)
    kagglehub.login()
    root = kagglehub.dataset_download(DATASET)

colon_dir = next(p.parent for p in Path(root).rglob("colon_aca") if p.is_dir())
print("Colon images found in:", colon_dir)
for cls in ["colon_aca", "colon_n"]:
    print(cls, len(list((colon_dir / cls).glob("*"))), "images")

## 3. Split into train / validation / test

In [ ]:
!python scripts/split_data.py --src "{colon_dir}" --out data/splits.csv

## 4. Train (about 10 to 15 minutes on a T4)

In [ ]:
!python -m src.train --csv data/splits.csv --arch resnet18 --epochs 8 --batch-size 32 --workers 2

## 5. Evaluate on the test set

In [ ]:
!python -m src.evaluate --csv data/splits.csv --ckpt outputs/best_model.pt
from IPython.display import Image, display
display(Image("outputs/confusion_matrix.png", width=420), Image("outputs/roc_curve.png", width=420))

## 6. Your results
Copy the block below and send it to Claude so the README and portfolio show your real numbers.

In [ ]:
import json
m = json.load(open("outputs/metrics.json"))
print("=== COLON CANCER DETECTION RESULTS ===")
for k in ["accuracy", "precision", "recall_sensitivity", "specificity", "f1", "roc_auc"]:
    print(f"{k:20s} {m[k]:.4f}")
print(f"{'n_test':20s} {m['n_test']}")

## 7. Pick sample images for the demo
Two benign and two cancer images from the test set, so visitors can try the demo without their own slides.

In [ ]:
import cv2, pandas as pd, shutil
df = pd.read_csv("data/splits.csv")
test = df[df.split == "test"]
ex = Path("examples"); shutil.rmtree(ex, ignore_errors=True); ex.mkdir()
for label, name in [(0, "benign"), (1, "adenocarcinoma")]:
    for i, path in enumerate(test[test.label == label].sample(2, random_state=7).path, 1):
        img = cv2.resize(cv2.imread(path), (384, 384), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(ex / f"{name}_{i}.jpeg"), img, [cv2.IMWRITE_JPEG_QUALITY, 90])
print(sorted(p.name for p in ex.iterdir()))

## 8. Publish the live demo to Hugging Face Spaces
Paste your Hugging Face **write** token when asked. The Space builds in about 5 minutes after upload.

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi
import shutil

api = HfApi(token=getpass("Hugging Face write token: "))
user = api.whoami()["name"]
space_id = f"{user}/colon-cancer-detection"
api.create_repo(space_id, repo_type="space", space_sdk="docker", exist_ok=True)

stage = Path("/content/space"); shutil.rmtree(stage, ignore_errors=True); stage.mkdir()
shutil.copytree("src", stage / "src", ignore=shutil.ignore_patterns("__pycache__"))
shutil.copytree("examples", stage / "examples")
(stage / "outputs").mkdir()
for f in ["best_model.pt", "metrics.json"]:
    shutil.copy(f"outputs/{f}", stage / "outputs" / f)
shutil.copy("app.py", stage / "app.py")
for f in ["Dockerfile", "requirements.txt", "README.md"]:
    shutil.copy(f"deploy/huggingface/{f}", stage / f)

api.upload_folder(folder_path=str(stage), repo_id=space_id, repo_type="space",
                  commit_message="Deploy colon cancer detection demo")
print("Live demo (ready in about 5 minutes):")
print(f"https://huggingface.co/spaces/{space_id}")

## 9. Download your results
Saves a zip with your metrics and plots. Send it to Claude to add them to GitHub.

In [ ]:
!cd outputs && zip -q ../colon_results.zip metrics.json history.json confusion_matrix.png roc_curve.png
from google.colab import files
files.download("colon_results.zip")